#  Data cleaning
**Part 1 (Cells 1–10):** structural cleaning (HTML, headers, links → placeholders), length cap (5,000 chars), removal of dataset give-away words, de-duplication. Saves:
- `phishing_general_clean.csv` – **training data for model**
- `phishing_combined_clean.csv`, `phishing_education_clean.csv`

**Part 2 (Cells 11–13):** investigation of the EduPhish dataset. Found it is a keyword-filtered subset of the same public corpora used for training (confirmed by its README), with many near-copies of training emails and little genuine education content. **EduPhish is therefore not used as a test set.** These cells are kept as evidence for that decision.

In [1]:
# Connect to Drive and load the files from data_download.ipynb

from google.colab import drive
drive.mount('/content/drive')

import os
import re                  # re = "regular expressions", a tool for finding text patterns
import numpy as np
import pandas as pd

folder = "/content/drive/MyDrive/Colab Notebooks/Master's project/Data"

combined = pd.read_csv(os.path.join(folder, "phishing_combined.csv"))
education = pd.read_csv(os.path.join(folder, "phishing_education.csv"))

print("Combined rows before cleaning:", len(combined))
print("Education rows before cleaning:", len(education))

Mounted at /content/drive
Combined rows before cleaning: 105353
Education rows before cleaning: 16942


In [2]:
# Measure the leakage BEFORE cleaning (did this before in data_explore.ipynb but I will do it again anyway to be sure and have it in front of me)
# I train a quick model to guess which dataset an email came from.
# A high score means the datasets are easy to tell apart, which is bad.
# I wrap this in a function so I can run the exact same test again after cleaning.

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

def source_guess_score(table, text_column):
    """Returns how often (0 to 1) a quick model can guess the source from the text."""
    train_text, test_text, train_src, test_src = train_test_split(
        table[text_column], table["source"], test_size=0.2, random_state=42, stratify=table["source"])
    tfidf = TfidfVectorizer(max_features=20000, stop_words="english")
    model = LogisticRegression(max_iter=1000)
    model.fit(tfidf.fit_transform(train_text), train_src)
    return model.score(tfidf.transform(test_text), test_src)

score_before = source_guess_score(combined, "text")
print(f"BEFORE cleaning: source can be guessed {score_before*100:.1f}% of the time")


BEFORE cleaning: source can be guessed 87.8% of the time


In [14]:
# Structural cleaning - a function that tidies one email
# Each step below removes clutter that is not part of the message itself.
# I replace links and email addresses with placeholder words instead of deleting them, because "this email contains a link" is a useful phishing clue.

def clean_text(text):
    text = str(text)

    # 1. Remove HTML tags such as <br>, <div>, <a href=...>
    text = re.sub(r"<[^>]+>", " ", text)

    # 2. Remove email header lines that give the dataset away
    #    e.g. "Message-ID: ...", "X-Mailer: ...", "Received: ...", "Content-Type: ..."
    text = re.sub(r"^(message-id|x-[\w-]+|received|content-type|content-transfer-encoding|"
                  r"mime-version|return-path|delivered-to|list-id|precedence|in-reply-to|"
                  r"references|sender|reply-to|date|to|cc|bcc|from):.*$",
                  " ", text, flags=re.IGNORECASE | re.MULTILINE)

    # 3. Replace web links with the word URL (keeps the clue, drops the dataset-specific address)
    text = re.sub(r"(https?://\S+|www\.\S+)", " URL ", text)

    # 4. Replace email addresses with the word EMAIL
    text = re.sub(r"\S+@\S+", " EMAIL ", text)

    # 5. Replace long runs of numbers (phone numbers, IDs, dates) with NUMBER
    text = re.sub(r"\d{5,}", " NUMBER ", text)

    # 5b. Replace any year (1990-2029) with the word YEAR - years reveal when a dataset was collected
    text = re.sub(r"\b(19[9]\d|20[0-2]\d)\b", " YEAR ", text)

    # 6. Remove leftover odd characters, but KEEP punctuation that carries meaning
    #    (! and ? and $ and % are often used in phishing)
    text = re.sub(r"[^\w\s!?$%£€.,:;'\"()\-]", " ", text)

    # 7. Squash multiple spaces / new lines into one space
    text = re.sub(r"\s+", " ", text).strip()

    return text

# Quick test on one made-up email so I can see what the function does
example = ("Message-ID: <12345@mail.example.com>\nFrom: it-support@school.org\n"
           "<p>URGENT!! Your password expires in 24 hours. Click http://bit.ly/reset-now "
           "or call 07700900123</p>")
print("BEFORE:", example)
print("AFTER: ", clean_text(example))

BEFORE: Message-ID: <12345@mail.example.com>
From: it-support@school.org
<p>URGENT!! Your password expires in 24 hours. Click http://bit.ly/reset-now or call 07700900123</p>
AFTER:  URGENT!! Your password expires in 24 hours. Click URL or call NUMBER


In [15]:
# Apply structural cleaning to every email.
# I keep the ORIGINAL text in the "text" column and put the cleaned version in a new "text_clean" column. Later, the model will learn from text_clean, but I can still use the original for things like counting links or capital letters.

combined["text_clean"] = combined["text"].apply(clean_text)
education["text_clean"] = education["text"].apply(clean_text)
print("Structural cleaning done.")


Structural cleaning done.


In [16]:
# I need to set length caps
# Very long emails are usually forwarded chains or code and slow everything down.
# Very short ones have nothing to learn from.
#   - emails longer than MAX_CHARS are cut down to their first MAX_CHARS characters
#   - emails shorter than MIN_CHARS are dropped

# Why these numbers (from data_explore):
#   - half of all emails are under 749 characters, three quarters under 2,018, so 5,000 leaves the typical email untouched and only trims the long tail
#   - the longest "email" was 4.6 MILLION characters, which is clearly not an email but a dump of something else
#   - the shortest was 3 characters, so a minimum of 20 removes near-empty rows

MAX_CHARS = 5000
MIN_CHARS = 20

def apply_length_cap(table):
    before = len(table)
    table = table[table["text_clean"].str.len() >= MIN_CHARS].copy()       # drop too-short
    dropped = before - len(table)
    truncated = (table["text_clean"].str.len() > MAX_CHARS).sum()
    table["text_clean"] = table["text_clean"].str.slice(0, MAX_CHARS)      # cut too-long
    print(f"  dropped {dropped} emails shorter than {MIN_CHARS} characters")
    print(f"  cut {truncated} emails down to {MAX_CHARS} characters")
    return table.reset_index(drop=True)

print("Combined:")
combined = apply_length_cap(combined)
print("Education:")
education = apply_length_cap(education)




Combined:
  dropped 0 emails shorter than 20 characters
  cut 5500 emails down to 5000 characters
Education:
  dropped 0 emails shorter than 20 characters
  cut 2448 emails down to 5000 characters


In [23]:
# 0 emails were dropped so I will have a second look

orig_len = education["text"].str.len()
print("Education rows:", len(education))
print("Over 5,000 chars in the original, by label:")
print(education[orig_len > 5000]["label"].value_counts())

# Look at the end of one long education email - what is filling the space?
long_one = education[orig_len > 5000].iloc[0]["text"]
print("\nFIRST 400:", long_one[:400])
print("\nLAST 400: ", long_one[-400:])

Education rows: 16679
Over 5,000 chars in the original, by label:
label
1    2062
0     782
Name: count, dtype: int64

FIRST 400: Market Watch News Flash
We are following the strength of our last pick with a company that 
deserves your immediate attention. Our |ast profi|e on Jan 18 featured 
ALMI 
at .44.
Our estimate of 1.0O has been exceeded with its recent high of 1.O7. 
However, this is not the time to rest on our recent achievements.

Significant short term trading exp|osion is being predicted in a unique 
and dynamic 

LAST 400:  e information provided in this 
newsletter as a starting point for gathering additional information on 
the 
profi|ed company to allow the investor to form their own opinion 
regarding investment.



If you wish to stop future mailings, or if you fee| you have been 
wrongful|y placed in our membership, p|ease go here or send a b|ank  
e mai| with No Thanks in the subject to
(-stox0032@ yahoo.com-)


In [17]:
# Leakage removal - find "give-away" words automatically
# A give-away word is one that appears in a large share of ONE source's emails but almost never anywhere else. Examples might be "enron", "linguist", a mailing-list name, or a footer sentence.

# IMPORTANT: I only search the six general sources from Dataset 1 and the synthetic dataset. I do NOT search the EduPhish source, because its "give-away" words would be things like "student" and "university" - that is the real education vocabulary I WANT the model to see.

# The two numbers below control how strict the search is:
#   INSIDE  = word must appear in at least this share of the source's emails
#   OUTSIDE = word must appear in at most this share of all other emails


from sklearn.feature_extraction.text import CountVectorizer

INSIDE = 0.10     # 10% of the source's emails. (I tried a couple numbers here to see what I get.)
OUTSIDE = 0.02     # 2% of everyone else's

general = combined[combined["source"] != "eduphish"]

vec = CountVectorizer(binary=True, min_df=30, lowercase=True)
X = vec.fit_transform(general["text_clean"])
words = np.array(vec.get_feature_names_out())

giveaway = {}                                  # source -> list of give-away words
for source in general["source"].unique():
    in_source = (general["source"] == source).values
    share_inside = np.asarray(X[in_source].mean(axis=0)).ravel()
    share_outside = np.asarray(X[~in_source].mean(axis=0)).ravel()
    found = words[(share_inside >= INSIDE) & (share_outside <= OUTSIDE)]
    giveaway[source] = sorted(found.tolist())
    print(f"\n{source}: {len(found)} give-away words")
    print("  ", giveaway[source][:40], "..." if len(found) > 40 else "")


SpamAssasin: 8 give-away words
   ['ilug', 'irish', 'maintainer', 'removed', 'sf', 'sponsored', 'subscription', 'un'] 

Enron: 5 give-away words
   ['attached', 'cc', 'ect', 'enron', 'www'] 

CEAS_08: 1 give-away words
   ['python'] 

Nazario: 11 give-away words
   ['administrator', 'confirm', 'emails', 'jose', 'kindly', 'mailbox', 'monkey', 'org', 'password', 'upgrade', 'verify'] 

Nigerian_Fraud: 152 give-away words
   ['abroad', 'according', 'accounts', 'africa', 'african', 'along', 'arrangement', 'aside', 'assist', 'assistance', 'attn', 'attorney', 'balance', 'bank', 'banking', 'behalf', 'beneficiary', 'benefit', 'bless', 'children', 'civil', 'claim', 'communication', 'confidence', 'confidentiality', 'contacted', 'contacting', 'contract', 'contractor', 'country', 'crash', 'death', 'deceased', 'decided', 'deposit', 'deposited', 'died', 'direct', 'director', 'discovered'] ...

Ling: 45 give-away words
   ['abstract', 'abstracts', 'ac', 'areas', 'book', 'committee', 'copies', 'deadli

In [18]:
# I have to decide which give-away words to remove

# The cell above found words that are concentrated in one source. Many of them are  GENUINE content and must stay, for example:
#   - Nigerian_Fraud words like "beneficiary", "fund", "deceased" are real advance-fee fraud vocabulary
#   - Nazario words like "verify", "password", "confirm" are real credential-phishing vocabulary
#   - Ling words like "students", "deadline", "paper" are real academic vocabulary that school email also contains

# Only words that identify a DATASET rather than a kind of email are removed.
# The decisions are listed here by source so they can be reported in the write-up.

# Rule used: remove a word only if it identifies the collection rather that the kind of email; keep any word a real school inbox could plausibly contain.

REMOVE = {
    # SpamAssassin: mailing-list footer text (Irish Linux Users Group, SourceForge)
    "2002", "ilug", "irish", "maintainer", "removed", "sf", "sponsored", "subscription", "un",
    # Enron: company name, internal address fragments, years, header clutter
    "enron", "2000", "2001", "ect", "hou", "cc", "www",
    # CEAS_08: collection year and one company's newsletter identity
    "2008", "cnn", "warner", "cable", "atlanta", "georgia", "aug",
    # Nazario: the collector's own email address
    "jose", "monkey", "org",
    # Ling: mailing-list identity and linguistics jargon (general academic words are kept)
    "1995", "1998", "edu", "english", "languages",
    "linguist", "linguists", "linguistic", "linguistics", "semantics", "syntax", "grammar",
    "discourse", "theoretical", "abstract", "abstracts", "ac", "tel",
    # Synthetic: leaked prompt word and character names the generator reused
    "keywords",
    "avery", "brown", "cameron", "casey", "jordan", "khan", "kim", "patel", "singh",
}

print("Words to remove:", len(REMOVE))

# Save the removed list for the dissertation appendix
pd.DataFrame({"word": sorted(REMOVE)}).to_csv(
    os.path.join(folder, "table_removed_giveaway_words.csv"), index=False)

# Build one pattern that matches any of these as a whole word (case does not matter)
pattern = re.compile(r"\b(" + "|".join(re.escape(w) for w in sorted(REMOVE)) + r")\b",
                     flags=re.IGNORECASE)

def remove_giveaways(text):
    text = pattern.sub(" ", text)
    return re.sub(r"\s+", " ", text).strip()

combined["text_clean"] = combined["text_clean"].apply(remove_giveaways)
education["text_clean"] = education["text_clean"].apply(remove_giveaways)
print("Give-away words removed.")

Words to remove: 54
Give-away words removed.


In [19]:
# I need to remove duplicates again
# Cleaning can make two slightly different emails identical, so I de-duplicate once more on the cleaned text.

before = len(combined)
combined = combined.drop_duplicates(subset="text_clean").reset_index(drop=True)
print("Combined: removed", before - len(combined), "duplicates after cleaning")

before = len(education)
education = education.drop_duplicates(subset="text_clean").reset_index(drop=True)
print("Education: removed", before - len(education), "duplicates after cleaning")

Combined: removed 6 duplicates after cleaning
Education: removed 0 duplicates after cleaning


In [20]:
# Measure the leakage AFTER cleaning
# Same test as in the second cell. The score should be lower now. It will not reach pure chance, because different sources really do contain different kinds of emails, but a big drop shows the obvious shortcuts are gone.

score_after = source_guess_score(combined, "text_clean")
print(f"BEFORE cleaning: source guessed {score_before*100:.1f}% of the time")
print(f"AFTER  cleaning: source guessed {score_after*100:.1f}% of the time")
print(f"Pure chance would be about {100 / combined['source'].nunique():.1f}%")


BEFORE cleaning: source guessed 87.8% of the time
AFTER  cleaning: source guessed 87.7% of the time
Pure chance would be about 12.5%


In [21]:
# It is weird that i got the same number before and after cleaning...
# Did the cleaning actually change the text?
print("Rows where text and text_clean differ:", (combined["text"] != combined["text_clean"]).sum(), "of", len(combined))

# Are removed words really gone?
for w in ["enron", "ilug", "linguist", "jose", "keywords"]:
    still_there = combined["text_clean"].str.contains(r"\b" + w + r"\b", case=False, regex=True).sum()
    print(f"'{w}' still appears in {still_there} cleaned emails")

# Show one Enron email before and after
row = combined[combined["source"] == "Enron"].iloc[0]
print("\nBEFORE:", row["text"][:300])
print("\nAFTER: ", row["text_clean"][:300])

Rows where text and text_clean differ: 98634 of 99273
'enron' still appears in 0 cleaned emails
'ilug' still appears in 0 cleaned emails
'linguist' still appears in 0 cleaned emails
'jose' still appears in 0 cleaned emails
'keywords' still appears in 0 cleaned emails

BEFORE: hpl nom for may 25 , 2001 ( see attached file : hplno 525 . xls )
- hplno 525 . xls

AFTER:  hpl nom for may 25 , YEAR ( see attached file : hplno 525 . xls ) - hplno 525 . xls


In [24]:
# Final summary and save

print("=== COMBINED (clean) ===")
print("Rows:", len(combined))
print("\nLabel balance (0 = legitimate, 1 = phishing):")
print(combined["label"].value_counts())
print("\nRows per source:")
print(combined["source"].value_counts())

# The general file = combined minus the EduPhish rows
general = combined[combined["source"] != "eduphish"].reset_index(drop=True)

print("\n=== GENERAL (clean) - used for training ===")
print("Rows:", len(general))
print("\nLabel balance:")
print(general["label"].value_counts())

print("\n=== EDUCATION (clean) - used for testing ===")
print("Rows:", len(education))
print("\nLabel balance:")
print(education["label"].value_counts())

combined.to_csv(os.path.join(folder, "phishing_combined_clean.csv"), index=False)
general.to_csv(os.path.join(folder, "phishing_general_clean.csv"), index=False)
education.to_csv(os.path.join(folder, "phishing_education_clean.csv"), index=False)
print("\nSaved -> phishing_combined_clean.csv, phishing_general_clean.csv, phishing_education_clean.csv")

=== COMBINED (clean) ===
Rows: 99273

Label balance (0 = legitimate, 1 = phishing):
label
1    49650
0    49623
Name: count, dtype: int64

Rows per source:
source
CEAS_08               34309
Enron                 29406
eduphish              12464
kuladeep_synthetic     9676
SpamAssasin            5764
Nigerian_Fraud         3255
Ling                   2857
Nazario                1542
Name: count, dtype: int64

=== GENERAL (clean) - used for training ===
Rows: 86809

Label balance:
label
1    43981
0    42828
Name: count, dtype: int64

=== EDUCATION (clean) - used for testing ===
Rows: 16679

Label balance:
label
0    9101
1    7578
Name: count, dtype: int64

Saved -> phishing_combined_clean.csv, phishing_general_clean.csv, phishing_education_clean.csv


In [25]:
# Check what is actually in the EduPhish dataset
# A sample email in the fifth cell's check turned out to be old stock spam, not an education email. This cell checks (1) how many EduPhish emails mention education at all, and (2) how many are near-copies of general-data emails.

import re

# 1. How many EduPhish emails actually mention education at all?
edu_words = r"\b(school|college|university|student|students|teacher|staff|campus|"\
            r"pupil|pupils|headteacher|principal|lecturer|faculty|tuition|exam|"\
            r"semester|term|classroom|parent|parents|moodle|canvas|blackboard)\b"
has_edu = education["text"].str.contains(edu_words, case=False, regex=True)
print("EduPhish emails mentioning any education word:", has_edu.sum(), "of", len(education))
print("By label:")
print(pd.crosstab(education["label"], has_edu, rownames=["label"], colnames=["mentions education"]))

# 2. Near-duplicate overlap with the general data: compare the first 200 characters
def key(t):
    return re.sub(r"\s+", " ", str(t).lower())[:200]
general_keys = set(general["text"].apply(key))
overlap = education["text"].apply(key).isin(general_keys)
print("\nEduPhish emails whose first 200 chars match a general-data email:", overlap.sum())
print("By label:")
print(education[overlap]["label"].value_counts())

/tmp/ipykernel_3156/3095147261.py:10: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  has_edu = education["text"].str.contains(edu_words, case=False, regex=True)


EduPhish emails mentioning any education word: 4476 of 16679
By label:
mentions education  False  True 
label                           
0                    6616   2485
1                    5587   1991

EduPhish emails whose first 200 chars match a general-data email: 4514
By label:
label
0    2399
1    2115
Name: count, dtype: int64


In [26]:
# I need to build two cleaner education test sets becayse the check above found two problems with the EduPhish dataset (16,679 emails):

#   1. CONTAMINATION - 4,514 emails (27%) start with exactly the same 200 characters as an email in my general training data. These are near- copies. If I left them in, the model would be tested on emails it had already learned from, and the score would look better than it really is.

#   2. NOT REALLY EDUCATION - only 4,476 emails (27%) mention an education word such as "school", "student" or "teacher". The other 73% could be about anything, so the dataset name "education-targeted" only describe  part of it.

# So I make two versions of the education test set and report both:
#   - "decontaminated" = EduPhish with the 4,514 near-copies removed.
#                        This is the MAIN test set.
#   - "strict"         = decontaminated AND mentions at least one education word.
#                        Smaller, but the only part I can honestly call education-targeted. Used as a second, tougher test.

edu_words = r"\b(?:school|college|university|student|students|teacher|staff|campus|"\
            r"pupil|pupils|headteacher|principal|lecturer|faculty|tuition|exam|"\
            r"semester|term|classroom|parent|parents|moodle|canvas|blackboard)\b"

general_keys = set(general["text"].apply(key))
is_copy = education["text"].apply(key).isin(general_keys)
mentions_edu = education["text"].str.contains(edu_words, case=False, regex=True)

edu_decontaminated = education[~is_copy].reset_index(drop=True)
edu_strict = education[~is_copy & mentions_edu].reset_index(drop=True)

for name, table in [("DECONTAMINATED", edu_decontaminated), ("STRICT", edu_strict)]:
    print(f"\n=== EDUCATION {name} ===")
    print("Rows:", len(table))
    print("Label balance (0 = legitimate, 1 = phishing):")
    print(table["label"].value_counts())

edu_decontaminated.to_csv(os.path.join(folder, "phishing_education_decontaminated_clean.csv"), index=False)
edu_strict.to_csv(os.path.join(folder, "phishing_education_strict_clean.csv"), index=False)
print("\nSaved both education test sets.")


=== EDUCATION DECONTAMINATED ===
Rows: 12165
Label balance (0 = legitimate, 1 = phishing):
label
0    6702
1    5463
Name: count, dtype: int64

=== EDUCATION STRICT ===
Rows: 3201
Label balance (0 = legitimate, 1 = phishing):
label
0    1752
1    1449
Name: count, dtype: int64

Saved both education test sets.


In [28]:
#Sample 5 of each

print("--- 5 STRICT phishing emails ---")
for t in edu_strict[edu_strict["label"] == 1]["text"].sample(5, random_state=2):
    print("*", t[:400].replace("\n", " "), "\n")

print("--- 5 emails EXCLUDED from strict (no education word) ---")
excluded = education[~is_copy & ~mentions_edu]
for t in excluded[excluded["label"] == 1]["text"].sample(5, random_state=2):
    print("*", t[:400].replace("\n", " "), "\n")

--- 5 STRICT phishing emails ---
* what have you been up to ? i don ' t know about you , but i am sick and tired of going down to the store to find out that your printer cartridges cost more than the printer itself ! i know how you feel - i print over 500 pages a day , and it feels like there is a vacuum sucking money out of my wallet ! now , this has got to stop because i know it doesn ' t cost the printer companies anywhere near  

* My name is Julie :)  This whole semester I felt like some excitement was missing... had brilliant idea  to have my first website featuring me (pics and videos).  It's neet how my private life is inside one website ;) Verify your age and connect to my webcam today -) Come I want to share my life experience with you ;)  http://nufwvxymdfc.glasswortnet.com/ju1/  ;)                         rem me http 

* CASTLE  INTERNATIONAL-LOTTERY.CO.UK                                                               Castle International Lottery                             

In [29]:
# Stronger near-copy check that ignores formatting
# The earlier check compared the first 200 characters as written. Some EduPhish emails look like reformatted copies (lowercase, spaces around punctuation), which that check would miss. This version strips everything except letters and numbers before comparing the first 40 words, so formatting cannot hide a copy.

def strong_key(t):
    t = re.sub(r"[^a-z0-9 ]", " ", str(t).lower())    # keep only letters, numbers, spaces
    return " ".join(t.split()[:40])                     # first 40 words

general_strong = set(general["text"].apply(strong_key))
is_copy_strong = education["text"].apply(strong_key).isin(general_strong)

print("Near-copies found by the strong check:", is_copy_strong.sum(), "of", len(education))
print("By label:")
print(education[is_copy_strong]["label"].value_counts())
print("\nOf the STRICT education set, how many are copies?",
      (is_copy_strong & ~is_copy & mentions_edu).sum(), "additional")

Near-copies found by the strong check: 4741 of 16679
By label:
label
0    2517
1    2224
Name: count, dtype: int64

Of the STRICT education set, how many are copies? 16 additional
